In [1]:
import json
import csv
import unicodedata

# ── Load files ────────────────────────────────────────────────────────────────
with open("normalization_map.json", encoding="utf-8") as f:
    norm_map = json.load(f)

# Sort longest-key-first so multi-char sequences (ज्ञ, क्ष, श्र …) are
# matched before their component characters are touched.
sorted_keys = sorted(norm_map.keys(), key=len, reverse=True)

# Characters that appear in replacement VALUES — these must not be stripped
# when they appear as standalone keys later in the sorted list.
protected_in_values = set("".join(norm_map.values()))


def normalize(text: str) -> str:
    # 1. NFC so composed/decomposed Devanagari variants are consistent
    text = unicodedata.normalize("NFC", text)
    # 2. Apply substitutions in longest-key-first order.
    #    If a key maps to "" (strip) but the character also appears in some
    #    replacement value (e.g. ् in ग्य), skip the strip — the longer
    #    conjunct rules already handled correct cases.
    for key in sorted_keys:
        replacement = norm_map[key]
        if replacement == "" and key in protected_in_values:
            continue          # don't strip chars needed inside replacements
        if key in text:
            text = text.replace(key, replacement)
    # 3. Collapse extra whitespace left by stripped characters
    text = " ".join(text.split())
    return text

In [2]:
# ── Read ──────────────────────────────────────────────────────────────────────
rows = []
with open("dataset.csv", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

In [3]:
# ── Normalize & write ─────────────────────────────────────────────────────────
out_path = "normalized_dataset.csv"
with open(out_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["original_text", "normalized_text", "label"])
    writer.writeheader()
    for row in rows:
        writer.writerow({
            "original_text":  row["text"],
            "normalized_text": normalize(row["text"]),
            "label":          row["label"],
        })

print(f"✅  Done — {len(rows)} rows written to {out_path}\n")

✅  Done — 705 rows written to normalized_dataset.csv



In [4]:
# ── Verification ──────────────────────────────────────────────────────────────
checks = {
    "व → ब":        ("व", "ब"),
    "श → स":        ("श", "स"),
    "ष → स":        ("ष", "स"),
    "ई → इ":        ("ई", "इ"),
    "ऊ → उ":        ("ऊ", "उ"),
    "ी → ि":        ("ी", "ि"),
    "ू → ु":        ("ू", "ु"),
    "ृ → रि":       ("ृ", "रि"),
    "ं stripped":   ("ं", ""),
    "। stripped":   ("।", ""),
    ", stripped":   (",", ""),
    "( ) stripped": ("(", ""),
    "ज्ञ → ग्य":   ("ज्ञ", "ग्य"),
    "क्ष → छ्य":   ("क्ष", "छ्य"),
    "श्र → सर":    ("श्र", "सर"),
    "त्र → तिर":   ("त्र", "तिर"),
}

print("=" * 60)
print("VERIFICATION CHECKS")
print("=" * 60)
all_pass = True
for label, (src, tgt) in checks.items():
    result = normalize(src)
    ok = result == tgt
    if not ok:
        all_pass = False
    print(f"  {'✅' if ok else '❌'}  {label:20s}  got: {result!r}  expected: {tgt!r}")

print()
print("All checks passed! ✅" if all_pass else "Some checks FAILED ❌")

print("\n" + "=" * 60)
print("SAMPLE OUTPUT (first 5 rows)")
print("=" * 60)
with open(out_path, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= 5:
            break
        print(f"\n[{row['label']}]")
        print(f"  ORIG : {row['original_text']}")
        print(f"  NORM : {row['normalized_text']}")

VERIFICATION CHECKS
  ✅  व → ब                 got: 'ब'  expected: 'ब'
  ✅  श → स                 got: 'स'  expected: 'स'
  ✅  ष → स                 got: 'स'  expected: 'स'
  ✅  ई → इ                 got: 'इ'  expected: 'इ'
  ✅  ऊ → उ                 got: 'उ'  expected: 'उ'
  ✅  ी → ि                 got: 'ि'  expected: 'ि'
  ✅  ू → ु                 got: 'ु'  expected: 'ु'
  ✅  ृ → रि                got: 'रि'  expected: 'रि'
  ✅  ं stripped            got: ''  expected: ''
  ✅  । stripped            got: ''  expected: ''
  ✅  , stripped            got: ''  expected: ''
  ✅  ( ) stripped          got: ''  expected: ''
  ✅  ज्ञ → ग्य             got: 'ग्य'  expected: 'ग्य'
  ✅  क्ष → छ्य             got: 'छ्य'  expected: 'छ्य'
  ✅  श्र → सर              got: 'सर'  expected: 'सर'
  ✅  त्र → तिर             got: 'तिर'  expected: 'तिर'

All checks passed! ✅

SAMPLE OUTPUT (first 5 rows)

[high]
  ORIG : शरीरबाट निरन्तर रगत बगेको रिपोर्ट गरिएको छ
  NORM : सरिरबाट निरन्तर रगत बगएकओ रिपओर्ट ग